# English to Arabic Translation with PyTorch Transformers

This notebook demonstrates how to fine-tune a transformer model to translate English text to Arabic. The code:

1. Loads English-Arabic sentence pairs from Kaggle
2. Fine-tunes a pre-trained translation model
3. Evaluates translation quality using BLEU score (higher is better)
4. Compares the fine-tuned model with the base model
5. Provides an interactive translation function

## Exercises for Students

This notebook contains several exercises where you'll need to complete key components:

1. Implementing a custom PyTorch Dataset for translation data
2. Completing the model training function 
3. Building a text translation function
4. Creating an interactive translation interface

Look for the `# TODO` comments throughout the notebook that indicate code you need to implement.


In [1]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import os
import kagglehub
from tqdm.notebook import tqdm
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction
import nltk
from torch.utils.data import Dataset, DataLoader
from transformers import MarianTokenizer, MarianMTModel, AdamW, get_linear_schedule_with_warmup

2025-03-16 21:51:38.018338: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-16 21:51:38.235896: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1742151098.311997   18644 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1742151098.335898   18644 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-16 21:51:38.522529: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [3]:
nltk.download('punkt_tab') 
nltk.download('perluniprops')
nltk.download('nonbreaking_prefixes')

[nltk_data] Downloading package punkt_tab to /home/ali/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package perluniprops to /home/ali/nltk_data...
[nltk_data]   Package perluniprops is already up-to-date!
[nltk_data] Downloading package nonbreaking_prefixes to
[nltk_data]     /home/ali/nltk_data...
[nltk_data]   Package nonbreaking_prefixes is already up-to-date!


True

In [4]:
# Download the dataset
print("Downloading dataset...")
path = kagglehub.dataset_download("samirmoustafa/arabic-to-english-translation-sentences")
print("Path to dataset files:", path)

# Check the downloaded files
print("Files in the dataset directory:")
files = os.listdir(path)
print(files)

Path to dataset files: /home/ali/.cache/kagglehub/datasets/samirmoustafa/arabic-to-english-translation-sentences/versions/1
Files in the dataset directory:
['ara_eng.txt']


In [5]:
# Read the dataset
with open(path + "/ara_eng.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()

In [6]:
# Display a few examples
print("Sample data:")
for i in range(10):
    print(lines[i])

Sample data:
Hi.	مرحبًا.

Run!	اركض!

Help!	النجدة!

Jump!	اقفز!

Stop!	قف!

Go on.	داوم.

Go on.	استمر.

Hello!	مرحباً.

Hurry!	تعجّل!

Hurry!	استعجل!



In [7]:
# Preprocess the data
data = []
for line in lines:
    parts = line.strip().split('\t')
    if len(parts) == 2:
        english, arabic = parts[0], parts[1]
        # Remove trailing period from Arabic if present
        arabic = arabic.rstrip('.')
        data.append((english, arabic))

In [8]:
# Create a DataFrame
df = pd.DataFrame(data, columns=['english', 'arabic'])
print(f"Dataset size: {len(df)} pairs")

Dataset size: 24638 pairs


In [9]:
# Display some statistics
print(f"Average English sentence length: {df['english'].apply(len).mean():.2f} characters")
print(f"Average Arabic sentence length: {df['arabic'].apply(len).mean():.2f} characters")

Average English sentence length: 102.01 characters
Average Arabic sentence length: 86.75 characters


In [10]:
# Split the data
train_df, test_val_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(test_val_df, test_size=0.5, random_state=42)
print(f"Training set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")

Training set size: 19710
Validation set size: 2464
Test set size: 2464


In [11]:
# Load pre-trained model and tokenizer for English to Arabic as starting point
model_name = "Helsinki-NLP/opus-mt-en-ar"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
model.to(device)

/home/ali/.virtualenvs/PyTorchENV/lib/python3.10/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


MarianMTModel(
  (model): MarianModel(
    (shared): Embedding(62802, 512, padding_idx=62801)
    (encoder): MarianEncoder(
      (embed_tokens): Embedding(62802, 512, padding_idx=62801)
      (embed_positions): MarianSinusoidalPositionalEmbedding(512, 512)
      (layers): ModuleList(
        (0-5): 6 x MarianEncoderLayer(
          (self_attn): MarianAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation_fn): SiLU()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
          (final_layer_norm): LayerNorm((512,), eps=1e-05

In [ ]:
# Exercise: Create a custom dataset for translation
class TranslationDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=128):
        # TODO: Initialize the dataset with dataframe, tokenizer, and max_length
        # YOUR CODE HERE
        pass
        
    def __len__(self):
        # TODO: Return the length of the dataset
        # YOUR CODE HERE
        pass
    
    def __getitem__(self, idx):
        # TODO: Extract the English and Arabic text from the dataframe at the given index
        # YOUR CODE HERE
        english_text = None  # Replace with your code
        arabic_text = None   # Replace with your code
        
        # TODO: Tokenize the English input text
        # Use the tokenizer with max_length, padding='max_length', truncation=True, and return_tensors='pt'
        # YOUR CODE HERE
        source_encoding = None  # Replace with your code
        
        # TODO: Tokenize the Arabic target text
        # Use the same parameters as above
        # YOUR CODE HERE
        target_encoding = None  # Replace with your code
        
        # TODO: Extract input_ids and attention_mask from source_encoding
        # and labels from target_encoding
        # Remember to use squeeze() to remove extra dimensions
        # YOUR CODE HERE
        input_ids = None      # Replace with your code
        attention_mask = None # Replace with your code
        labels = None         # Replace with your code
        
        # TODO: Replace padding token id with -100 in labels
        # This ensures padding tokens are ignored in loss computation
        # YOUR CODE HERE
        
        # TODO: Return a dictionary with input_ids, attention_mask, and labels
        # YOUR CODE HERE
        return {}

In [ ]:
# Create datasets and dataloaders
batch_size = 32  # Adjust based on your GPU memory

train_dataset = TranslationDataset(train_df, tokenizer)
val_dataset = TranslationDataset(val_df, tokenizer)
test_dataset = TranslationDataset(test_df, tokenizer)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size)

In [ ]:
# Define training function
def train_model(model, train_dataloader, val_dataloader, epochs=3, learning_rate=5e-5):
    # Prepare optimizer and scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    total_steps = len(train_dataloader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, 
        num_warmup_steps=0,
        num_training_steps=total_steps
    )
    
    # Track training progress
    train_losses = []
    val_losses = []
    
    # Training loop
    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        
        # Training
        model.train()
        train_loss = 0
        train_progress_bar = tqdm(train_dataloader, desc="Training")
        
        for batch in train_progress_bar:
            # Move batch to device
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            # TODO: Zero the gradients
            # YOUR CODE HERE
            
            # TODO: Perform forward pass with the model
            # Hint: Use model() with input_ids, attention_mask, and labels as parameters
            # YOUR CODE HERE
            outputs = None
            
            # TODO: Get the loss from outputs
            # YOUR CODE HERE
            loss = None
            
            train_loss += loss.item()
            
            # TODO: Perform backward pass and update parameters
            # Hint: Call backward() on the loss and step() on optimizer and scheduler
            # YOUR CODE HERE
            
            # Update progress bar
            train_progress_bar.set_postfix({'loss': loss.item()})
        
        avg_train_loss = train_loss / len(train_dataloader)
        train_losses.append(avg_train_loss)
        print(f"Average training loss: {avg_train_loss:.4f}")
        
        # Validation
        # TODO: Set the model to evaluation mode
        # YOUR CODE HERE
        
        val_loss = 0
        val_progress_bar = tqdm(val_dataloader, desc="Validation")
        
        # TODO: Use torch.no_grad() context manager for validation
        # YOUR CODE HERE
            for batch in val_progress_bar:
                # Move batch to device
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)
                
                # Forward pass
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )
                
                loss = outputs.loss
                val_loss += loss.item()
                
                # Update progress bar
                val_progress_bar.set_postfix({'loss': loss.item()})
        
        avg_val_loss = val_loss / len(val_dataloader)
        val_losses.append(avg_val_loss)
        print(f"Average validation loss: {avg_val_loss:.4f}")
    
    # Plot training progress
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Training Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    return model, train_losses, val_losses

In [ ]:
# Train the model
print("Starting model fine-tuning...")
epochs = 3  # Increase for better results, decrease for faster training
fine_tuned_model, train_losses, val_losses = train_model(model, train_dataloader, val_dataloader, epochs=epochs)

In [ ]:
# Save the fine-tuned model
output_dir = "./fine_tuned_en_ar_model"
os.makedirs(output_dir, exist_ok=True)
fine_tuned_model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model saved to {output_dir}")

In [ ]:
# Function to translate text
def translate_text(text, model, tokenizer):
    # TODO: Set the model to evaluation mode
    # YOUR CODE HERE
    
    # TODO: Tokenize the text with padding, truncation and max_length=128
    # Hint: Use tokenizer() with return_tensors="pt"
    # YOUR CODE HERE
    inputs = None
    
    # Move to device
    input_ids = inputs['input_ids'].to(device)
    attention_mask = inputs['attention_mask'].to(device)
    
    # TODO: Generate translation using model.generate()
    # Hint: Use with torch.no_grad(): and set appropriate parameters
    # YOUR CODE HERE
    outputs = None
    
    # TODO: Decode the generated tokens to get the translated text
    # Hint: Use tokenizer.batch_decode() with skip_special_tokens=True
    # YOUR CODE HERE
    translated_text = None
    
    # Return the first (and only) translation in the batch
    return translated_text[0]

In [ ]:
# Test the translation function
sample_texts = df['english'].iloc[:50].tolist()
print("\nSample translations with fine-tuned model:")
for text in sample_texts:
    translation = translate_text(text, fine_tuned_model, tokenizer)
    print(f"English: {text}")
    print(f"Arabic (translated): {translation}")
    print()

In [ ]:
# Calculate BLEU score
def calculate_bleu(references, hypotheses):
    def tokenize_text(text):
        # Simple tokenization by whitespace
        return text.split()
    
    # Tokenize reference and hypothesis
    references_tokenized = [tokenize_text(ref) for ref in references]
    hypotheses_tokenized = [tokenize_text(hyp) for hyp in hypotheses]
    
    # Calculate BLEU score
    smoothing = SmoothingFunction().method1
    
    # Individual BLEU scores
    individual_scores = []
    for ref, hyp in zip(references_tokenized, hypotheses_tokenized):
        score = sentence_bleu([ref], hyp, smoothing_function=smoothing)
        individual_scores.append(score)
    
    # Corpus BLEU score
    corpus_score = corpus_bleu([[r] for r in references_tokenized], hypotheses_tokenized, 
                              smoothing_function=smoothing)
    
    return individual_scores, corpus_score

In [ ]:
# Evaluate on test set
def evaluate_model(model, test_data, tokenizer, num_samples=100):
    # Limit evaluation to a subset for speed
    test_subset = test_data.iloc[:num_samples]
    
    references = []
    hypotheses = []
    
    model.eval()
    print(f"Evaluating model on {len(test_subset)} samples...")
    for _, row in tqdm(test_subset.iterrows(), total=len(test_subset)):
        english_text = row['english']
        true_arabic = row['arabic']
        
        # Translate
        predicted_arabic = translate_text(english_text, model, tokenizer)
        
        references.append(true_arabic)
        hypotheses.append(predicted_arabic)
    
    # Calculate BLEU scores
    individual_scores, corpus_score = calculate_bleu(references, hypotheses)
    
    print(f"Corpus BLEU score: {corpus_score:.4f}")
    print(f"Average sentence BLEU score: {np.mean(individual_scores):.4f}")
    
    # Display some examples
    print("\nSample predictions:")
    for i in range(min(5, len(references))):
        print(f"English: {test_subset.iloc[i]['english']}")
        print(f"True Arabic: {references[i]}")
        print(f"Predicted Arabic: {hypotheses[i]}")
        print(f"BLEU score: {individual_scores[i]:.4f}")
        print()
    
    return corpus_score, individual_scores, references, hypotheses

In [ ]:
# Run the evaluation
print("Evaluating fine-tuned model...")
corpus_score, individual_scores, references, hypotheses = evaluate_model(fine_tuned_model, test_df, tokenizer, num_samples=50)

In [ ]:
# Plot the distribution of BLEU scores
plt.figure(figsize=(10, 6))
plt.hist(individual_scores, bins=20, alpha=0.7, color='blue')
plt.axvline(corpus_score, color='red', linestyle='dashed', linewidth=2, label=f'Corpus BLEU: {corpus_score:.4f}')
plt.axvline(np.mean(individual_scores), color='green', linestyle='dashed', linewidth=2, 
           label=f'Mean Sentence BLEU: {np.mean(individual_scores):.4f}')
plt.title('Distribution of BLEU Scores (Fine-tuned Model)')
plt.xlabel('BLEU Score')
plt.ylabel('Frequency')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Compare with base model
print("\nComparing with base model (no fine-tuning)...")
base_model = MarianMTModel.from_pretrained(model_name).to(device)
base_corpus_score, base_individual_scores, base_references, base_hypotheses = evaluate_model(base_model, test_df, tokenizer, num_samples=50)


In [ ]:
# Plot comparison of BLEU scores
plt.figure(figsize=(12, 6))
plt.hist(base_individual_scores, bins=20, alpha=0.5, color='red', label='Base Model')
plt.hist(individual_scores, bins=20, alpha=0.5, color='blue', label='Fine-tuned Model')
plt.axvline(base_corpus_score, color='red', linestyle='dashed', linewidth=2, 
           label=f'Base Corpus BLEU: {base_corpus_score:.4f}')
plt.axvline(corpus_score, color='blue', linestyle='dashed', linewidth=2, 
           label=f'Fine-tuned Corpus BLEU: {corpus_score:.4f}')
plt.title('Comparison of BLEU Scores: Base vs. Fine-tuned Model')
plt.xlabel('BLEU Score')
plt.ylabel('Frequency')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Interactive translation function
def translate_interactive():
    # TODO: Create a loop that continues until the user enters 'quit'
    # YOUR CODE HERE
    while True:
        # TODO: Get input text from the user
        # Hint: Use input() function with a prompt
        # YOUR CODE HERE
        text = None
        
        # TODO: Check if the user wants to quit
        # Hint: Compare text.lower() with 'quit'
        # YOUR CODE HERE
        
        # TODO: Translate text using both models
        # Hint: Call translate_text() with each model
        # YOUR CODE HERE
        ft_translation = None
        base_translation = None
        
        # TODO: Print the translations
        # YOUR CODE HERE
        
        print()  # Empty line for better readability

In [ ]:
translate_interactive()